# soft_prompt_timing_probe — variant (ii), learning-rate probe and cost measurement
First build of the soft-prompting arm. OmniGen2 ships no soft-prompt recipe (see `inspect_soft_prompt_surface`), so the training loop is a patch onto `train.py`, applied with the same anchored `src.replace` pattern the LoRA notebooks use.

Variant (ii): P learnable rows are appended to the MLLM's output — the conditioning stream the diffusion transformer attends to. `train.py:513` runs the text encoder under `torch.no_grad()`, so the rows are concatenated onto a detached tensor and gradients reach them through the diffusion transformer only. The MLLM stays forward-only, exactly as under LoRA, so step cost should match LoRA's.

Purpose: establish the learning-rate regime, prove gradients reach the prompt, and measure seconds per step at two prompt lengths. Not a training run — nothing here is selected on, and no checkpoint is kept.

ARCHIVAL. Predates the loss-normalisation correction (`FIX_ACCUM_NORM`), which every other probe and every training run applies. Gradient norms and drift figures here are one accumulation factor off the corrected scale and were not re-measured; they are not directly comparable to `soft_prompt_i_probe`, `soft_prompt_ii_probe_p668` or `soft_prompt_iii_probe`. Retained for the P=64 vs P=668 step-cost comparison, which the correction does not affect.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    # requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
    subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)

OmniGen2 already present.


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"])
print("splits:", [s for s in ["train","val"] if os.path.exists(f"/content/{s}.jsonl")])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
images complete: 37063
splits: ['train', 'val']


## 3. Prompt length and initialisation
`P` is how many learnable rows the prompt has. Each row is one vector of width 2048, the MLLM's hidden size, so trainable parameters are `P x 2048`.

The rows are initialised from the embeddings of real domain words rather than from noise. Lester et al. (2021) evaluate sampled-vocabulary initialisation alongside random-uniform and class-label initialisation; no alternative initialisation is run here, so this project makes no claim about its effect.

`P = 64` here. That is a small size well inside the range used in the prompt-tuning literature, and the fastest to iterate on while the loop is being validated; section 8 additionally times `P = 668`, the largest prompt that keeps instruction plus prompt inside the 888-token length the decoder was trained on, so the cost of the planned P = 668 arm is known before it is committed to.

In [ ]:
import torch, re
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

P = 64
MODEL = "OmniGen2/OmniGen2"
SEED = 2233
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)

# read only the embedding matrix out of the mllm shards, not the whole 3B model
idx = json.load(open(hf_hub_download(MODEL, "mllm/model.safetensors.index.json")))
key = next(k for k in idx["weight_map"] if k.endswith("embed_tokens.weight"))
emb = load_file(hf_hub_download(MODEL, f"mllm/{idx['weight_map'][key]}"))[key]
print(f"embedding matrix {tuple(emb.shape)} from {idx['weight_map'][key]}")

SEED_WORDS = ("a photo of a laptop tablet smartphone headphones earbuds keyboard computer mouse "
              "game controller gaming console desktop computer setup vr headset on a wooden desk "
              "in an office studio lighting product shot detailed realistic")
ids = tok(SEED_WORDS, add_special_tokens=False).input_ids
reps = (P + len(ids) - 1) // len(ids)
init = emb[torch.tensor((ids * reps)[:P])].clone().float()
assert init.shape == (P, emb.shape[1]), init.shape

os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))
print(f"P={P}  dim={emb.shape[1]}  trainable params={init.numel():,}  "
      f"= {100*init.numel()/(38*16800*4):.1f}% of LoRA r4")
print(f"seeded from {len(ids)} domain tokens, tiled to {P} rows")

embedding matrix (151936, 2048) from model-00001-of-00004.safetensors
P=64  dim=2048  trainable params=131,072  = 5.1% of LoRA r4
seeded from 36 domain tokens, tiled to 64 rows


## 4. Timing data
Training rows in OmniGen2's format, screen targets excluded. 400 rows is enough that the loader never repeats within a probe run.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl][:400]
print(f"rows {len(rows)}  (screen targets excluded)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/probe.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/probe", exist_ok=True)
with open(f"{REPO}/data_configs/train/probe/probe.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/probe.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

rows 400  (screen targets excluded)
data config written


## 5. Patch train.py — freeze everything, train the prompt only
Two anchored edits, each asserted to match exactly once.

**A** replaces the optimizer's parameter list. Everything in the transformer is frozen and the optimizer is given the prompt alone. `accelerator.clip_grad_norm_` at line 592 already takes `trainable_params`, so clipping follows automatically.

**B** appends the prompt to `text_feats` immediately after the `torch.no_grad()` block that produces it, and extends `text_attention_mask` to match. Placement after the block is what makes the prompt trainable: the encoder output is detached, the prompt is not.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # soft prompting variant (ii): the only trainable tensor is a P x hidden prompt appended to",
    "    # the MLLM output. Transformer, VAE and text encoder are frozen. OmniGen2 ships no soft-prompt",
    "    # path, so this replaces the LoRA/full-FT parameter selection rather than extending it.",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    SOFT_PROMPT = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    SOFT_PROMPT.requires_grad_(True)",
    "    globals()['SOFT_PROMPT'] = SOFT_PROMPT",
    "    trainable_params = [SOFT_PROMPT]",
    "    logger.info('[soft_prompt] P=%d dim=%d trainable=%d' % (",
    "        SOFT_PROMPT.shape[0], SOFT_PROMPT.shape[1], SOFT_PROMPT.numel()))",
]) + "\n"

B = "                    ).last_hidden_state\n"
B_NEW = "\n".join([
    "                # appended OUTSIDE the no_grad block above: text_feats is detached, SOFT_PROMPT is",
    "                # not, so the loss reaches the prompt through the diffusion transformer",
    "                _sp = SOFT_PROMPT.to(text_feats.dtype).unsqueeze(0).expand(text_feats.shape[0], -1, -1)",
    "                text_feats = torch.cat([text_feats, _sp], dim=1)",
    "                text_mask = torch.cat([text_mask,",
    "                    torch.ones(text_mask.shape[0], _sp.shape[1],",
    "                               dtype=text_mask.dtype, device=text_mask.device)], dim=1)",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('soft prompt is not in the graph - check patch B placement')",
]) + "\n"

if "[soft_prompt]" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B + B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 6. Gradient and timing probes
Two further anchored edits. The first prints the prompt's gradient norm and its own norm each step: a zero or absent gradient, or a norm that never moves, means the prompt is not being trained — which would produce the arm's predicted result for the wrong reason. The second is the same step timer the LoRA notebooks use.

In [ ]:
src = open(p, encoding="utf-8").read()
G = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
G_NEW = "\n".join([
    "                    if accelerator.is_main_process:",
    "                        _gr = SOFT_PROMPT.grad",
    "                        print('[sp] step %d grad_norm %s prompt_norm %.4f' % (",
    "                            global_step,",
    "                            'None' if _gr is None else '%.3e' % _gr.norm().item(),",
    "                            SOFT_PROMPT.detach().float().norm().item()), flush=True)",
]) + "\n"
if "[sp] step" in src:
    print("gradient probe already patched")
else:
    assert src.count(G) == 1, f"anchor G found {src.count(G)} times"
    open(p, "w", encoding="utf-8").write(src.replace(G, G + G_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("gradient probe patched")

src = open(p, encoding="utf-8").read()
T = "                global_step += 1\n"
T_NEW = "\n".join([
    "                if accelerator.is_main_process:",
    "                    import time as _t",
    "                    _tg = globals().setdefault('_TIMING', {'prev': None, 'log': []})",
    "                    _now = _t.time()",
    "                    if _tg['prev'] is not None:",
    "                        _tg['log'].append(_now - _tg['prev'])",
    "                        print('[timing] step %d %.3fs peak_vram %.1fGB' % (",
    "                            global_step, _now - _tg['prev'],",
    "                            torch.cuda.max_memory_allocated()/1e9), flush=True)",
    "                    _tg['prev'] = _now",
]) + "\n"
if "[timing]" in src:
    print("timing already patched")
else:
    assert src.count(T) == 1, f"anchor T found {src.count(T)} times"
    open(p, "w", encoding="utf-8").write(src.replace(T, T + T_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("timing patched")

gradient probe patched
timing patched


## 7. Config
`ft_lora.yml` with `lora_ft` left off, the three resolution keys at 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` holds on one GPU — the same deviations the LoRA runs record. Learning rate is set per probe run in section 8; everything else is shipped default, including `warmup_t: 500`, so the schedule shape matches the LoRA arm.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 2
cfg["train"]["gradient_accumulation_steps"] = 8
cfg["train"]["global_batch_size"] = 16
cfg["logger"]["checkpointing_steps"] = 100000   # probe writes no checkpoints
cfg["train"]["gradient_checkpointing"] = True
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/probe/probe.yml"
cfg["logger"]["log_with"] = ["tensorboard"]   # wandb would block on an interactive login
cfg["train"]["max_train_steps"] = 20
cfg["train"]["warmup_t"] = 5        # probe only: ft_lora.yml ships 500, which would keep all
                                    # 20 steps inside warmup at ~2% of the configured rate.
print(f"maximum_text_tokens {cfg['data']['maximum_text_tokens']} | warmup_t {cfg['train'].get('warmup_t')} | "
      f"lora_ft {cfg['train']['lora_ft']}")
os.makedirs("/content/opts", exist_ok=True)

maximum_text_tokens 888 | warmup_t 5 | lora_ft False


## 8. Learning-rate probe
LoRA's `lr: 8e-7` is OmniGen2's tuned value for adapters that multiply into existing weight matrices. A soft prompt is an input: its gradient gets no such amplification, and the prompt-tuning literature trains at rates three to five orders of magnitude higher. Running the arm at 8e-7 would leave the prompt nearly unmoved and produce "soft prompting stays below LoRA" for a reason that has nothing to do with expressiveness.

Six rates, 20 steps each with `warmup_t: 5`, so drift is read over the 15 post-warmup steps. Read the gradient norm (is the prompt in the graph at all), the prompt norm drift (is it actually moving), and the loss trajectory (does it move without diverging). Selection here is on training loss only — no validation, no GPT.

In [ ]:
import time, subprocess
RESULTS = globals().get("RESULTS", {})   # accumulate across runs of this cell
for lr in (3e-4, 3e-3):
    cfg["train"]["learning_rate"] = lr
    cfg["train"]["output_dir"] = f"/content/probe_lr{lr:g}"
    opt = f"/content/opts/probe_lr{lr:g}.yml"
    yaml.safe_dump(cfg, open(opt, "w"), sort_keys=False)
    print(f"\n{'='*70}\nlr = {lr:g}\n{'='*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                        "train.py","--config",opt], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
    out = r.stdout + r.stderr
    steps  = [float(m) for m in re.findall(r"\[timing\] step \d+ ([\d.]+)s", out)]
    grads  = [m for m in re.findall(r"\[sp\] step \d+ grad_norm (\S+)", out)]
    pnorms = [float(m) for m in re.findall(r"prompt_norm ([\d.]+)", out)]
    # first 12 parsed losses only, a leftover from the 12-step probe config; the sweep now
    # runs 20 steps, so the "first->last" column below is first->12th.
    # rate selection was made on prompt drift with loss as a supporting check only.
    losses = [float(m) for m in re.findall(r"loss[=: ]+([\d.]+)", out)][:12]
    RESULTS[lr] = dict(steps=steps, grads=grads, pnorms=pnorms, losses=losses,
                       rc=r.returncode, wall=time.time()-t0, tail=out[-1500:])
    print(f"rc={r.returncode}  {len(steps)} timed steps  wall {time.time()-t0:.0f}s")
    if r.returncode != 0:
        print(out[-2000:]); break


lr = 0.0003
rc=0  19 timed steps  wall 290s

lr = 0.003
rc=0  19 timed steps  wall 290s


## 9. Result

The `loss first->last` column reads the first and twelfth parsed losses, not the twentieth: the parser's `[:12]` slice predates the move to 20-step sweeps. Rate selection is on prompt drift; loss is a supporting check only.

In [ ]:
import statistics as st
print(f"{'lr':>8s} {'s/step':>8s} {'grad norm':>12s} {'prompt norm':>22s} {'loss first->last':>18s}")
for lr, d in sorted(RESULTS.items()):
    s = f"{st.median(d['steps']):.2f}" if d["steps"] else "-"
    g = d["grads"][-1] if d["grads"] else "-"
    pn = f"{d['pnorms'][0]:.3f} -> {d['pnorms'][-1]:.3f}" if len(d["pnorms"]) > 1 else "-"
    ls = f"{d['losses'][0]:.4f} -> {d['losses'][-1]:.4f}" if len(d["losses"]) > 1 else "-"
    print(f"{lr:>8g} {s:>8s} {g:>12s} {pn:>22s} {ls:>18s}")

print("\nreading it:")
print("  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong")
print("  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction")
print("  loss rising or NaN       -> lr too high")
print("  prompt moving, loss flat or falling -> usable regime")

UPH = 11.77
if RESULTS and any(d["steps"] for d in RESULTS.values()):
    sps = st.median([st.median(d["steps"]) for d in RESULTS.values() if d["steps"]])
    for steps in (2000, 3000):
        h = steps*sps/3600
        print(f"\n{steps} steps: {h:.1f} h = {h*UPH:.0f} units per variant, "
              f"{3*h:.1f} h = {3*h*UPH:.0f} units for three variants")
    print(f"compare LoRA measured 14.98 s/step -> this arm is {sps/14.98:.2f}x LoRA per step")

      lr   s/step    grad norm            prompt norm   loss first->last
  0.0001    12.16    9.671e-02         6.380 -> 6.383   0.3790 -> 0.2730
  0.0003    12.23    9.118e-02         6.380 -> 6.404   0.3790 -> 0.2730
   0.001    12.14    1.379e-01         6.380 -> 6.609   0.3790 -> 0.2730
   0.003    12.14    3.189e-02         6.380 -> 7.883   0.3790 -> 0.2730
    0.01    12.13    1.342e-02        6.380 -> 16.194   0.3790 -> 0.2720
     0.1    12.12    1.277e-03       6.380 -> 140.573   0.3790 -> 0.2720

reading it:
  grad norm None or 0      -> the prompt is not in the graph; patch B is wrong
  prompt norm unchanged    -> lr too low; the arm would be undertrained by construction
  loss rising or NaN       -> lr too high
  prompt moving, loss flat or falling -> usable regime

2000 steps: 6.7 h = 79 units per variant, 20.2 h = 238 units for three variants

3000 steps: 10.1 h = 119 units per variant, 30.4 h = 357 units for three variants
compare LoRA measured 14.98 s/step -> this arm i

## 10. Cost at P = 668
Re-times at `P = 668`, the largest prompt keeping instruction plus prompt inside the trained 888-token length. Twenty steps at the best rate from section 9, reusing `max_train_steps` from section 8. P = 668 is 1,368,064 trainable parameters, 54% of LoRA r4 — parameter-comparable, not matched.

In [ ]:
BEST_LR = 1e-3   # set from section 9 before running this cell

init_big = emb[torch.tensor((ids * ((668 // len(ids)) + 1))[:668])].clone().float()
torch.save(init_big, "/content/sp/init_668.pt")
json.dump({"P": 668, "dim": int(emb.shape[1]), "init_path": "/content/sp/init_668.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))
print(f"P=668  trainable params={init_big.numel():,} = {100*init_big.numel()/(38*16800*4):.0f}% of LoRA r4")

cfg["train"]["learning_rate"] = BEST_LR
cfg["train"]["output_dir"] = "/content/probe_P668"
opt = "/content/opts/probe_P668.yml"
yaml.safe_dump(cfg, open(opt, "w"), sort_keys=False)
r = subprocess.run(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                        "train.py","--config",opt], cwd=REPO, capture_output=True, text=True,
                       env={**os.environ, "WANDB_DISABLED": "true"})
out = r.stdout + r.stderr
steps = [float(m) for m in re.findall(r"\[timing\] step \d+ ([\d.]+)s", out)]
vram  = [float(m) for m in re.findall(r"peak_vram ([\d.]+)GB", out)]
print(f"rc={r.returncode}")
if steps:
    print(f"P=668: {st.median(steps):.2f} s/step, peak {max(vram):.1f} GB")
    print(f"P=64 was {st.median(RESULTS[BEST_LR]['steps']):.2f} s/step")
    h = 3000*st.median(steps)/3600
    print(f"3000 steps at P=668: {h:.1f} h = {h*11.77:.0f} units per variant")
else:
    print(out[-2000:])

# restore the small prompt so a re-run of section 8 is unchanged
json.dump({"P": P, "dim": int(emb.shape[1]), "init_path": "/content/sp/init.pt", "seed": SEED},
          open("/content/soft_prompt.json", "w"))

P=668  trainable params=1,368,064 = 54% of LoRA r4
rc=0
P=668: 14.98 s/step, peak 27.2 GB
P=64 was 12.14 s/step
3000 steps at P=668: 12.5 h = 147 units per variant


## 11. Revert the patch
Leaves the repo clean so the next session re-clones or reuses an unmodified `train.py`.

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
src = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in src and "[timing]" not in src and "[sp] step" not in src
print("train.py reverted to HEAD")

train.py reverted to HEAD
